# RT-DETR-L Training

## Config and Paths

In [ ]:
import os

DATASET_YAML = '/home/achin/COS40007-Group/dataset/dataset.yaml'
WEIGHTS_PATH = '/home/achin/COS40007-Group/rtdetr-l.pt'
PROJECT_DIR  = '/home/achin/COS40007-Group/runs/defect_detection'
DATASET_ROOT = '/home/achin/COS40007-Group/dataset'

CLASS_NAMES = ['crack', 'pothole', 'wall_peeling']
BACKGROUND_PREFIXES = ('wall-', 'road-')
NC = 3
RUN = 'rtdetr_l_v8'
SPLITS = ['train', 'val', 'test']

IMAGE_DIRS = {s: os.path.join(DATASET_ROOT, s, 'images') for s in SPLITS}
LABEL_DIRS = {s: os.path.join(DATASET_ROOT, s, 'labels') for s in SPLITS}

print("Paths configured:")
for s in SPLITS:
    all_imgs = [f for f in os.listdir(IMAGE_DIRS[s]) if f.endswith('.jpg')]
    bg_imgs  = [f for f in all_imgs if f.startswith(BACKGROUND_PREFIXES)]
    def_imgs = [f for f in all_imgs if not f.startswith(BACKGROUND_PREFIXES)]
    print(f"  {s}: {len(def_imgs)} defect + {len(bg_imgs)} background = {len(all_imgs)} total")

## Dataset Exploratory Data Analysis

In [ ]:
import os
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt

def count_images(directory):
    p = Path(directory)
    return len(list(p.glob('*.jpg')) + list(p.glob('*.jpeg')))

def count_labels(split):
    label_dir = Path(LABEL_DIRS[split])
    class_counts = Counter()
    bg_count = 0
    for txt in label_dir.glob('*.txt'):
        content = txt.read_text().strip()
        if not content:
            bg_count += 1
        else:
            for line in content.splitlines():
                if line.strip():
                    cls = int(line.split()[0])
                    class_counts[cls] += 1
    return class_counts, bg_count

for split in SPLITS:
    counts, bg_count = count_labels(split)
    total_images = count_images(IMAGE_DIRS[split])
    print(f"\n{split.upper()}: {total_images} images")
    for cls_id in sorted(counts):
        print(f"  {CLASS_NAMES[cls_id]}: {counts[cls_id]} annotations")
    print(f"  background (empty): {bg_count} images")
    ratio = bg_count / max(sum(counts.values()), 1)
    print(f"  bg/defect ratio: {ratio:.2f} {'⚠️ consider reducing' if ratio > 0.5 else '✅'}")

# Bar chart — defect classes only, background shown separately
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, split in zip(axes, SPLITS):
    counts, bg_count = count_labels(split)
    names  = [CLASS_NAMES[i] for i in sorted(counts)] + ['background']
    values = [counts[i] for i in sorted(counts)] + [bg_count]
    colors = ['steelblue', 'coral', 'mediumseagreen', 'lightgray']
    bars = ax.bar(names, values, color=colors)
    ax.set_title(f'{split.upper()} — Distribution')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(val), ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'class_distribution.png'), dpi=150)
plt.show()

## Visualise Sample Annotations

In [ ]:
import matplotlib.patches as patches
from PIL import Image

def visualize_samples_by_class(split, n_per_class=3):
    img_dir = Path(IMAGE_DIRS[split])
    lbl_dir = Path(LABEL_DIRS[split])
    COLORS  = ['red', 'coral', 'mediumseagreen']

    # defect rows + 1 background row
    total_rows = NC + 1
    fig, axes = plt.subplots(total_rows, n_per_class, figsize=(5 * n_per_class, 4 * total_rows))

    # --- defect class rows ---
    for row, cls_name in enumerate(CLASS_NAMES):
        cls_imgs = sorted(img_dir.glob(f'{cls_name}-*.jpg'))[:n_per_class]

        for col in range(n_per_class):
            ax = axes[row][col]
            ax.axis('off')

            if col >= len(cls_imgs):
                ax.set_title(f'{cls_name} — no sample', fontsize=9)
                continue

            img_path = cls_imgs[col]
            img = Image.open(img_path).convert('RGB')
            w, h = img.size
            ax.imshow(img)
            ax.set_title(f'{img_path.name}', fontsize=8)

            lbl_path = lbl_dir / (img_path.stem + '.txt')
            if lbl_path.exists():
                for line in lbl_path.read_text().strip().splitlines():
                    parts = line.split()
                    if not parts:
                        continue
                    c, cx, cy, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = (cx - bw/2) * w
                    y1 = (cy - bh/2) * h
                    rect = patches.Rectangle(
                        (x1, y1), bw*w, bh*h,
                        linewidth=2, edgecolor=COLORS[c % len(COLORS)], facecolor='none'
                    )
                    ax.add_patch(rect)
                    ax.text(x1, y1 - 4, CLASS_NAMES[c], color=COLORS[c % len(COLORS)],
                            fontsize=7, fontweight='bold',
                            bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))

    bg_imgs = []
    for prefix in BACKGROUND_PREFIXES:
        bg_imgs.extend(sorted(img_dir.glob(f'{prefix}*.jpg')))
    bg_imgs = bg_imgs[:n_per_class]

    for col in range(n_per_class):
        ax = axes[NC][col]
        ax.axis('off')
        if col < len(bg_imgs):
            img = Image.open(bg_imgs[col]).convert('RGB')
            ax.imshow(img)
            ax.set_title(f'{bg_imgs[col].name}\n(background — no labels)', fontsize=8)
        else:
            ax.set_title('background — no sample', fontsize=9)

    plt.suptitle(f'Sample Annotations — {split.upper()}', fontsize=13)
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, f'sample_annotations_{split}.png'), dpi=150)
    plt.show()

visualize_samples_by_class('train', n_per_class=3)

## Iterative Training — Setup

In [ ]:
from pathlib import Path

for split in SPLITS:
    cache = Path(LABEL_DIRS[split]).parent / 'labels.cache'
    if cache.exists():
        cache.unlink()
        print(f"Deleted cache: {cache}")
    else:
        print(f"No cache found for {split} — OK")

In [ ]:
from pathlib import Path
import random

SEED = 42
SPLIT_DIR = Path('/home/achin/COS40007-Group/dataset/splits')
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Cumulative train sizes for the 6 iterations
ITER_TRAIN_SIZES = [400, 428, 456, 484, 512, 541]


def make_splits(force: bool = True):
    """Generate per-iteration train lists + fixed val/test lists, deterministically.

    Writes:
      dataset/splits/iter{0..5}_train.txt  (cumulative train image paths)
      dataset/splits/val.txt               (184 paths, fixed)
      dataset/splits/test.txt              (181 paths, fixed)

    Idempotent: skips files that already exist unless `force=True`.
    Seed=42 => the same 400-image baseline + same batch order every run.
    """
    train_imgs = sorted(
        p for p in (Path(DATASET_ROOT) / 'train' / 'images').glob('*.jpg')
    )
    rng = random.Random(SEED)
    rng.shuffle(train_imgs)

    written = []
    for n, cum in enumerate(ITER_TRAIN_SIZES):
        out = SPLIT_DIR / f'iter{n}_train.txt'
        if out.exists() and not force:
            continue
        subset = train_imgs[:cum]
        out.write_text('\n'.join(str(p) for p in subset) + '\n', encoding='utf-8')
        written.append(str(out))

    for split in ('val', 'test'):
        out = SPLIT_DIR / f'{split}.txt'
        if out.exists() and not force:
            continue
        files = sorted((Path(DATASET_ROOT) / split / 'images').glob('*.jpg'))
        out.write_text('\n'.join(str(p) for p in files) + '\n', encoding='utf-8')
        written.append(str(out))

    print(f'make_splits: wrote {len(written)} files into {SPLIT_DIR}')
    for p in written:
        print(f'  {p}')


make_splits()


In [ ]:
def write_iter_data_yaml(n: int) -> str:
    """Emit dataset/dataset_iter{n}.yaml pointing train/val/test at the per-iter .txt files.

    Ultralytics accepts a .txt path under `train:` (one image path per line) so the
    iter loop can grow the train set without touching the underlying image directories.
    Returns the YAML path as a string.
    """
    p = Path(f'/home/achin/COS40007-Group/dataset/dataset_iter{n}.yaml')
    p.write_text(
        f"train: /home/achin/COS40007-Group/dataset/splits/iter{n}_train.txt\n"
        f"val:   /home/achin/COS40007-Group/dataset/splits/val.txt\n"
        f"test:  /home/achin/COS40007-Group/dataset/splits/test.txt\n"
        f"nc: {NC}\n"
        f"names: {CLASS_NAMES}\n",
        encoding='utf-8',
    )
    return str(p)


print(f'write_iter_data_yaml ready (caller passes n in 0..{len(ITER_TRAIN_SIZES) - 1})')


In [ ]:
from ultralytics import RTDETR
import json
import time
import random as _random
from pathlib import Path
import pandas as pd
from PIL import Image as _PILImage

iterlog_path = Path(PROJECT_DIR) / f'{RUN}_iterlog.csv'
if not iterlog_path.exists():
    iterlog_path.write_text(
        'iter,cum_train_imgs,new_imgs,mAP50,mAP50_95,precision,recall,'
        'pct_iou_gt_0_5,pct_iou_gt_0_9,best_epoch,time_h,notes\n',
        encoding='utf-8',
    )

stop_flag = Path(PROJECT_DIR) / 'STOP'
final_iter = None  # tracks the last successfully completed iter


def _read_first_xywh(label_path: Path):
    """Read the first YOLO-format normalized xywh box from a .txt. Returns None if absent."""
    if not label_path.exists():
        return None
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.split()
        if not parts:
            continue
        try:
            return (float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4]))
        except (ValueError, IndexError):
            continue
    return None


def _iou_xywh(box_a, box_b, w, h):
    """IoU between two normalized xywh boxes given image dims (w, h)."""
    ax1 = (box_a[0] - box_a[2] / 2) * w
    ay1 = (box_a[1] - box_a[3] / 2) * h
    ax2 = (box_a[0] + box_a[2] / 2) * w
    ay2 = (box_a[1] + box_a[3] / 2) * h
    bx1 = (box_b[0] - box_b[2] / 2) * w
    by1 = (box_b[1] - box_b[3] / 2) * h
    bx2 = (box_b[0] + box_b[2] / 2) * w
    by2 = (box_b[1] + box_b[3] / 2) * h
    inter_x1 = max(ax1, bx1); inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2); inter_y2 = min(ay2, by2)
    inter = max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def _iterlog_has_row(path: Path, n: int) -> bool:
    if not path.exists():
        return False
    for line in path.read_text(encoding='utf-8').splitlines()[1:]:
        if line.startswith(f'{n},'):
            return True
    return False


import torch
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

for n in range(len(ITER_TRAIN_SIZES)):
    cum = ITER_TRAIN_SIZES[n]
    new = cum - (ITER_TRAIN_SIZES[n-1] if n > 0 else 0)
    run_name = f'{RUN}_iter{n}'
    run_dir = Path(PROJECT_DIR) / run_name

    last_pt = run_dir / 'weights' / 'last.pt'
    if last_pt.exists() and _iterlog_has_row(iterlog_path, n):
        print(f'iter {n}: already complete (last.pt + iterlog row present), skipping')
        final_iter = n
        continue

    if stop_flag.exists():
        print(f'STOP sentinel present, exiting at iter {n}')
        break

    data_yaml = write_iter_data_yaml(n)
    if n == 0:
        model = RTDETR(WEIGHTS_PATH)
    else:
        prev_last = Path(PROJECT_DIR) / f'{RUN}_iter{n-1}' / 'weights' / 'best.pt'
        if not prev_last.exists():
            print(f'iter {n}: previous best.pt missing at {prev_last}; aborting')
            break
        model = RTDETR(str(prev_last))
    train_kwargs = dict(
        data=data_yaml,
        epochs=50, imgsz=640, batch=8,
        workers=4, amp=True, device=0,
        project=PROJECT_DIR, name=run_name, exist_ok=True,
        optimizer='AdamW', lr0=1e-4, lrf=0.001, weight_decay=1e-4,
        warmup_epochs=5, warmup_bias_lr=0.1, patience=50,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        fliplr=0.5, flipud=0.1, scale=0.5,
        mosaic=1.0, copy_paste=0.3, degrees=10.0, translate=0.1,
        save=True, plots=True,
    )

    t0 = time.time()
    print(f'\n=== iter {n}: training ({cum} train images, {new} new) ===')
    model.train(**train_kwargs)
    t_h = (time.time() - t0) / 3600.0

    # Val for mAP50 + per-class
    val = model.val(data=data_yaml, split='val', plots=False, save_json=False, verbose=False)
    map50 = float(val.box.map50); map50_95 = float(val.box.map)
    P = float(val.box.mp); R = float(val.box.mr)
    per_class = [float(x) for x in val.box.ap50]

    # Per-test-image IoU
    test_files = [Path(l) for l in SPLIT_DIR.joinpath('test.txt').read_text(encoding='utf-8').splitlines() if l.strip()]
    preds = model.predict(source=[str(p) for p in test_files], conf=0.001, save=False, verbose=False)
    iou_rows = []
    for img_p, r in zip(test_files, preds):
        with _PILImage.open(img_p) as im:
            w, h = im.size
        gt_n = _read_first_xywh(Path('/home/achin/COS40007-Group/dataset/test/labels') / (img_p.stem + '.txt'))
        if len(r.boxes) > 0:
            conf = float(r.boxes.conf[0])
            pred_xyxy = r.boxes.xyxy[0].cpu().numpy().tolist()
            pred_n = ((pred_xyxy[0]+pred_xyxy[2])/(2*w),
                      (pred_xyxy[1]+pred_xyxy[3])/(2*h),
                      (pred_xyxy[2]-pred_xyxy[0])/w,
                      (pred_xyxy[3]-pred_xyxy[1])/h)
        else:
            conf = 0.0; pred_xyxy = None; pred_n = None
        if gt_n is not None and pred_n is not None:
            iou = _iou_xywh(gt_n, pred_n, w, h)
            gt_xyxy = [(gt_n[0]-gt_n[2]/2)*w, (gt_n[1]-gt_n[3]/2)*h,
                       (gt_n[0]+gt_n[2]/2)*w, (gt_n[1]+gt_n[3]/2)*h]
        else:
            iou = 0.0; gt_xyxy = None
        iou_rows.append({
            'image': img_p.name, 'conf': conf,
            'pred_xyxy': pred_xyxy, 'gt_xyxy': gt_xyxy, 'iou': iou,
        })
    iou_csv = Path(PROJECT_DIR) / f'{run_name}_test_iou.csv'
    pd.DataFrame(iou_rows).to_csv(iou_csv, index=False)
    ious = [row['iou'] for row in iou_rows]
    pct_5 = sum(1 for i in ious if i > 0.5) / max(len(ious), 1)
    pct_9 = sum(1 for i in ious if i > 0.9) / max(len(ious), 1)

    # 2 sample images (random pick from test set)
    sample_dir = Path(PROJECT_DIR) / f'{run_name}_samples'
    sample_dir.mkdir(exist_ok=True)
    _random.seed(SEED + n)  # reproducible per-iter sample
    sample_imgs = _random.sample(test_files, min(2, len(test_files)))
    model.predict(source=[str(p) for p in sample_imgs], conf=0.25, save=True,
                  project=str(sample_dir), name='.', exist_ok=True)

    # Best epoch from results.csv
    best_epoch = ''
    res_csv = run_dir / 'results.csv'
    if res_csv.exists():
        try:
            res = pd.read_csv(res_csv)
            best_epoch = int(res['metrics/mAP50(B)'].idxmax()) + 1
        except Exception:
            pass

    # Per-iter summary JSON
    summary = {
        'iter': n, 'cum_train_imgs': cum, 'new_imgs': new,
        'mAP50': map50, 'mAP50_95': map50_95, 'precision': P, 'recall': R,
        'pct_iou_gt_0_5': pct_5, 'pct_iou_gt_0_9': pct_9,
        'best_epoch': best_epoch, 'time_h': t_h,
        'per_class_ap50': dict(zip(CLASS_NAMES, per_class)),
    }
    summary_path = Path(PROJECT_DIR) / f'{run_name}_summary.json'
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    # Append iterlog row
    with iterlog_path.open('a', encoding='utf-8') as f:
        f.write(f'{n},{cum},{new},{map50:.4f},{map50_95:.4f},{P:.4f},{R:.4f},'
                f'{pct_5:.4f},{pct_9:.4f},{best_epoch},{t_h:.2f},\n')

    print(f'iter {n} done: mAP50={map50:.4f} %IoU>0.5={pct_5:.3f} time={t_h:.2f}h')

    final_iter = n
    if map50 >= 0.85:
        stop_flag.touch()
        print(f'iter {n} hit mAP50 >= 0.85; STOP sentinel written')
        break

print(f'\nFinal completed iter: {final_iter}')


## Iterative Training — Loop

In [ ]:
import pandas as pd

iterlog_path = Path(PROJECT_DIR) / f'{RUN}_iterlog.csv'
BEST_ITER = None
BEST_RUN_NAME = None
BEST_WEIGHTS = None
if iterlog_path.exists():
    df = pd.read_csv(iterlog_path)
    if not df.empty:
        print('Iter log:')
        print(df.to_string(index=False))
        best_idx = df['mAP50'].idxmax()
        BEST_ITER = int(df.loc[best_idx, 'iter'])
        BEST_RUN_NAME = f'{RUN}_iter{BEST_ITER}'
        BEST_WEIGHTS = str(Path(PROJECT_DIR) / BEST_RUN_NAME / 'weights' / 'best.pt')
        print(f'\nBest iter: {BEST_ITER} (mAP50={df.loc[best_idx, "mAP50"]:.4f})')
        print(f'Best weights: {BEST_WEIGHTS}')
    else:
        print('iterlog is empty')
else:
    print(f'iterlog not found at {iterlog_path}')


## Iterative Training — Comparison Report


In [ ]:
import matplotlib.pyplot as plt

metrics = [
    ('train/giou_loss', 'Train giou_loss'),
    ('metrics/mAP50(B)', 'Val mAP50'),
    ('metrics/precision(B)', 'Val Precision'),
    ('metrics/recall(B)', 'Val Recall'),
]
fig, axes = plt.subplots(6, 4, figsize=(20, 24), sharex=True)
for row, n in enumerate(range(6)):
    res_csv = Path(PROJECT_DIR) / f'{RUN}_iter{n}' / 'results.csv'
    for col, (m, lbl) in enumerate(metrics):
        ax = axes[row][col]
        if not res_csv.exists():
            ax.text(0.5, 0.5, f'iter {n}: not run', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
            continue
        df = pd.read_csv(res_csv)
        if m in df.columns:
            ax.plot(df['epoch'], df[m], label=lbl)
            ax.set_title(f'iter {n} - {lbl}')
            ax.set_xlabel('epoch')
            ax.legend(loc='best', fontsize=8)
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, f'{m}\nnot in results.csv', ha='center', va='center', transform=ax.transAxes)
            ax.axis('off')
plt.tight_layout()
plt.savefig(Path(PROJECT_DIR) / 'iter_comparison_training_curves.png', dpi=120)
plt.show()


In [ ]:
import matplotlib.image as mpimg

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for n in range(6):
    r, c = n // 3, n % 3
    ax = axes[r][c]
    p = Path(PROJECT_DIR) / f'{RUN}_iter{n}' / 'confusion_matrix_normalized.png'
    if p.exists():
        ax.imshow(mpimg.imread(p))
        ax.set_title(f'iter {n} - Confusion Matrix (Normalized)')
    else:
        ax.text(0.5, 0.5, f'iter {n}: not run', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout()
plt.savefig(Path(PROJECT_DIR) / 'iter_comparison_confusion_matrices.png', dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for n in range(6):
    r, c = n // 3, n % 3
    ax = axes[r][c]
    p = Path(PROJECT_DIR) / f'{RUN}_iter{n}' / 'BoxPR_curve.png'
    if p.exists():
        ax.imshow(mpimg.imread(p))
        ax.set_title(f'iter {n} - PR Curve')
    else:
        ax.text(0.5, 0.5, f'iter {n}: not run', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')
plt.tight_layout()
plt.savefig(Path(PROJECT_DIR) / 'iter_comparison_pr_curves.png', dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(6, 2, figsize=(10, 24))
for n in range(6):
    sample_dir = Path(PROJECT_DIR) / f'{RUN}_iter{n}_samples'
    samples = sorted(sample_dir.glob('*.jpg'))[:2] if sample_dir.exists() else []
    for col in range(2):
        ax = axes[n][col]
        if col < len(samples):
            ax.imshow(mpimg.imread(samples[col]))
            ax.set_title(f'iter {n} - {samples[col].name}', fontsize=8)
        else:
            ax.text(0.5, 0.5, f'iter {n}: no sample', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
plt.tight_layout()
plt.savefig(Path(PROJECT_DIR) / 'iter_comparison_samples.png', dpi=120)
plt.show()


In [ ]:
print('Per-class AP@50 across iters:')
header = f'{"class":<15}' + ''.join(f'iter{i}'.rjust(10) for i in range(6))
print(header)
for cls in CLASS_NAMES:
    row = [cls.ljust(15)]
    for n in range(6):
        summary = Path(PROJECT_DIR) / f'{RUN}_iter{n}_summary.json'
        if summary.exists():
            v = json.loads(summary.read_text(encoding='utf-8'))['per_class_ap50'].get(cls)
            row.append(f'{v:.4f}'.rjust(10) if v is not None else '--'.rjust(10))
        else:
            row.append('--'.rjust(10))
    print(''.join(row))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for n in range(6):
    r, c = n // 3, n % 3
    ax = axes[r][c]
    iou_csv = Path(PROJECT_DIR) / f'{RUN}_iter{n}_test_iou.csv'
    if iou_csv.exists():
        df = pd.read_csv(iou_csv)
        ax.hist(df['iou'], bins=20, edgecolor='black', alpha=0.7)
        ax.set_title(f'iter {n} - Test IoU distribution')
        ax.set_xlabel('IoU')
        ax.set_ylabel('Count')
        ax.axvline(0.5, color='red', linestyle='--', label='IoU=0.5')
        ax.axvline(0.9, color='green', linestyle='--', label='IoU=0.9')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    else:
        ax.text(0.5, 0.5, f'iter {n}: not run', ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
plt.tight_layout()
plt.savefig(Path(PROJECT_DIR) / 'iter_comparison_iou_distribution.png', dpi=120)
plt.show()


## Final Model Evaluation — Best Iter


In [ ]:
if BEST_WEIGHTS is None or not Path(BEST_WEIGHTS).exists():
    raise RuntimeError(f'Best iter weights not found: {BEST_WEIGHTS}. Run the iter loop first.')
print(f'Loading best iter weights: {BEST_WEIGHTS}')
trained_model = RTDETR(BEST_WEIGHTS)
print(f'Loaded model from iter {BEST_ITER} (run: {BEST_RUN_NAME})')


## Evaluation on Test Set

In [ ]:
test_metrics = trained_model.val(
    data=DATASET_YAML,
    split='test',
    plots=True,
    project=PROJECT_DIR,
    name=f'{BEST_RUN_NAME}_test',
    exist_ok=True,
)

print("\n--- Test Metrics ---")
print(f"mAP@50:     {test_metrics.box.map50:.4f}")
print(f"mAP@50-95:  {test_metrics.box.map:.4f}")
print(f"Precision:  {test_metrics.box.mp:.4f}")
print(f"Recall:     {test_metrics.box.mr:.4f}")
print(f"\nPer-class AP@50:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {test_metrics.box.ap50[i]:.4f}")

## Training Curves

In [ ]:
import pandas as pd
import matplotlib.image as mpimg

results_csv = os.path.join(PROJECT_DIR, BEST_RUN_NAME, 'results.csv')
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

plots = [
    ('train/giou_loss',       'Train GIoU Loss',  'steelblue'),
    ('train/cls_loss',        'Train Cls Loss',   'coral'),
    ('train/l1_loss',         'Train L1 Loss',    'goldenrod'),
    ('val/giou_loss',         'Val GIoU Loss',    'navy'),
    ('val/cls_loss',          'Val Cls Loss',     'firebrick'),
    ('val/l1_loss',           'Val L1 Loss',      'darkorange'),
    ('metrics/mAP50(B)',      'mAP@50',           'green'),
    ('metrics/mAP50-95(B)',   'mAP@50-95',        'darkgreen'),
    ('metrics/precision(B)',  'Precision',        'purple'),
    ('metrics/recall(B)',     'Recall',           'orange'),
    ('lr/pg0',                'LR (backbone)',    'gray'),
    ('lr/pg1',                'LR (weights)',     'darkgray'),
]

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
for ax, (col, title, color) in zip(axes.flatten(), plots):
    if col in df.columns:
        ax.plot(df['epoch'], df[col], color=color, linewidth=1.5)
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)
    else:
        ax.set_title(f'{title} (not found)')
        ax.axis('off')

plt.suptitle('RT-DETR Training History', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'training_curves.png'), dpi=150)
plt.show()

## Inference and Visualisation on Test Images

In [ ]:
trained_model.predict(
    source=IMAGE_DIRS['test'],
    conf=0.25,
    iou=0.5,
    save=True,
    project=PROJECT_DIR,
    name=f'{BEST_RUN_NAME}_predictions',
    exist_ok=True,
)

pred_dir = Path(PROJECT_DIR) / f'{BEST_RUN_NAME}_predictions'

fig, axes = plt.subplots(NC, 3, figsize=(15, 4 * NC))
for row, cls_name in enumerate(CLASS_NAMES):
    cls_preds = sorted(pred_dir.glob(f'{cls_name}-*.jpg'))[:3]
    for col in range(3):
        ax = axes[row][col]
        ax.axis('off')
        if col < len(cls_preds):
            ax.imshow(mpimg.imread(cls_preds[col]))
            ax.set_title(cls_preds[col].name, fontsize=8)
        else:
            ax.set_title(f'{cls_name} — no prediction saved', fontsize=8)

plt.suptitle('RT-DETR Predictions — Test Set (by class)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'prediction_samples.png'), dpi=150)
plt.show()

## Display Evaluation Plots

In [ ]:
eval_plots = [
    ('confusion_matrix.png',            'Confusion Matrix'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix (Normalised)'),
    ('PR_curve.png',                    'Precision-Recall Curve'),
    ('P_curve.png',                     'Precision Curve'),
    ('R_curve.png',                     'Recall Curve'),
]

run_dir = Path(PROJECT_DIR) / BEST_RUN_NAME

for filename, title in eval_plots:
    p = run_dir / filename
    if p.exists():
        plt.figure(figsize=(10, 7))
        plt.imshow(mpimg.imread(p))
        plt.title(title, fontsize=13)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Not yet generated: {filename}")

## Inference Speed Benchmark

In [ ]:
import time

test_imgs = list(Path(IMAGE_DIRS['test']).glob('*.jpg'))[:50]

# Warmup
trained_model.predict(source=[str(test_imgs[0])], conf=0.25, verbose=False)

# Timed run
start = time.time()
trained_model.predict(source=[str(p) for p in test_imgs], conf=0.25, verbose=False)
elapsed = time.time() - start

fps = len(test_imgs) / elapsed
print(f"Images tested : {len(test_imgs)}")
print(f"Total time    : {elapsed:.2f}s")
print(f"FPS           : {fps:.1f}")
print(f"ms/image      : {1000/fps:.1f}")

## Display Evaluation Plots

In [ ]:
eval_plots = [
    ('confusion_matrix.png',            'Confusion Matrix'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix (Normalised)'),
    ('BoxPR_curve.png',                    'Precision-Recall Curve'),
    ('BoxP_curve.png',                     'Precision Curve'),
    ('BoxR_curve.png',                     'Recall Curve'),
]

run_dir = Path(PROJECT_DIR) / BEST_RUN_NAME

for filename, title in eval_plots:
    p = run_dir / filename
    if p.exists():
        plt.figure(figsize=(10, 7))
        plt.imshow(mpimg.imread(p))
        plt.title(title, fontsize=13)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Not yet generated: {filename}")